# 🧠 Laboratorio 1.3 – Advanced Boolean Retrieval and Positional Indexing

**Corso:** Information Retrieval – Laurea Magistrale in Informatica  
**Università:** Roma “Tor Vergata”  
**Docente:** Danilo Croce

---

## 🎯 Obiettivo del laboratorio

In questo laboratorio completiamo il sistema di retrieval booleano introdotto nel notebook precedente, trasformandolo in una versione più espressiva, più efficiente e più vicina alle scelte adottate nei sistemi di ricerca reali.

Nel laboratorio 1.2 abbiamo costruito un primo motore di ricerca booleano su una collezione reale, ma con alcune semplificazioni deliberate:
- uso di operazioni basate su `set`
- assenza di precedenza tra operatori
- assenza di parentesi
- gestione semplificata di `NOT`
- nessun supporto alle **phrase query** su collezione reale

In questo notebook affrontiamo questi limiti, ma con una scelta progettuale precisa e didatticamente utile:

## 📌 Scelta di design fondamentale

L’operatore `NOT` **non** verrà trattato come complemento dell’intera collezione.

Invece, `NOT` verrà interpretato come **filtro locale su un insieme candidato già positivo**.

### Esempi supportati
- `graphic AND file`
- `graphic AND NOT file`
- `graphic AND NOT "file format"`
- `(graphic AND NOT "file format") OR imag`

### Esempi non supportati
- `NOT graphic`
- `imag OR NOT graphic`
- `NOT "file format"`

Questa scelta ha quattro vantaggi:
- evita di costruire insiemi enormi di documenti
- rende il sistema più efficiente
- semplifica il query processing
- riflette un uso più realistico della negazione nei sistemi di ricerca

---

## 🧩 Linguaggio di query del laboratorio

Il sistema supporterà:

- termini singoli
- frasi tra virgolette, ad esempio `"file format"`
- operatori booleani `AND`, `OR`, `NOT`
- parentesi
- una struttura di valutazione controllata:
  - `OR` separa clausole a livello esterno
  - all’interno di ciascuna clausola, le unità sono combinate con `AND`
  - `NOT` può comparire solo come negazione locale di una unità all’interno di una clausola positiva

La semantica adottata sarà la seguente:

- una query è una **OR di clausole**
- ogni clausola è una **AND di unità**
- una unità può essere:
  - un termine
  - una frase tra virgolette
  - una negazione `NOT unit`
- ogni clausola deve contenere **almeno una unità positiva**

Questa scelta non coincide con un parser booleano generale, ma definisce un linguaggio di query più controllato e più semplice da elaborare.

---

## 🛠️ Struttura del notebook

Nel notebook procederemo in quattro passi principali:

1. definizione delle utility di preprocessing e indicizzazione  
2. definizione delle utility per il parsing e il query processing  
3. costruzione degli indici su una collezione reale  
4. esecuzione di query booleane e phrase query

---

## 🎓 Risultati di apprendimento attesi

Al termine del laboratorio lo studente dovrebbe essere in grado di:

- costruire un **inverted index** e un **positional index**
- distinguere tra retrieval unigram e retrieval basato su posizioni
- usare postings lists ordinate per `AND`, `OR` e filtraggio locale con `NOT`
- supportare query con termini, frasi e parentesi
- comprendere il ruolo della semantica del linguaggio di query nel comportamento del sistema

---

## 📌 Take-home message

Il punto fondamentale di questo laboratorio è che un sistema di Information Retrieval non dipende solo dai documenti e dai termini, ma anche da:

- come i documenti vengono pre-elaborati
- come l’indice viene costruito
- come la query viene interpretata
- quali restrizioni semantiche vengono adottate

Per questo motivo, progettare un sistema booleano significa progettare insieme:
- la **rappresentazione**
- l’**indicizzazione**
- il **linguaggio di query**
- il **query processing**

---

## Preparazione della collezione e delle strutture di base

Prima di eseguire query booleane e phrase query, dobbiamo costruire l’infrastruttura fondamentale del sistema.

In questa sezione ci occuperemo di tre passaggi essenziali:

- caricare una **collezione reale** di documenti
- applicare una pipeline di **preprocessing** ai testi
- costruire le strutture dati necessarie per il retrieval:
  - **inverted index**
  - **positional index**
  - **document frequency**

L’obiettivo è trasformare una raccolta di documenti grezzi in una rappresentazione adatta alla ricerca efficiente.

In [ ]:
import re
from collections import defaultdict

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
nltk.download('punkt_tab')

from sklearn.datasets import fetch_20newsgroups

# Risorse NLTK necessarie per:
# - tokenizzazione in parole
# - lista di stopwords inglesi
nltk.download("punkt")
nltk.download("stopwords")


# ============================================================
# 1. Load a real collection
# ============================================================

def load_collection(categories=None, subset="train"):
    """
    Carica un sottoinsieme del dataset 20 Newsgroups.

    Parameters
    ----------
    categories : list[str] or None
        Lista delle categorie da caricare. Se None, usiamo per default
        la categoria 'comp.graphics'.
    subset : str
        Porzione del dataset da usare, ad esempio 'train' oppure 'test'.

    Returns
    -------
    list[str]
        Lista dei documenti testuali grezzi.
    """
    if categories is None:
        categories = ["comp.graphics"]

    dataset = fetch_20newsgroups(
        subset=subset,
        categories=categories,
        remove=()  # non rimuoviamo automaticamente header, footer o quotes
    )
    return dataset.data


# ============================================================
# 2. Preprocessing utilities
# ============================================================

# Oggetti globali usati in più punti della pipeline di preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def remove_header(text):
    """
    Rimuove la parte iniziale del documento fino alla prima riga vuota.

    Nei documenti di 20 Newsgroups questa zona contiene spesso metadati
    come From, Subject, Organization, ecc.

    Se non viene trovata una separazione chiara tra header e body,
    il testo viene restituito invariato.
    """
    parts = text.split("\n\n", 1)
    if len(parts) == 2:
        return parts[1]
    return text

def convert_lower_case(text):
    """
    Converte tutto il testo in minuscolo per uniformare le forme lessicali.
    """
    return text.lower()

def convert_numbers(text):
    """
    Sostituisce le cifre numeriche con la corrispondente forma testuale.

    Esempio:
        '3d' -> ' three d'

    È una scelta didattica semplice per evitare di lasciare le cifre
    come simboli grezzi nel testo.
    """
    number_map = {
        "0": " zero ",
        "1": " one ",
        "2": " two ",
        "3": " three ",
        "4": " four ",
        "5": " five ",
        "6": " six ",
        "7": " seven ",
        "8": " eight ",
        "9": " nine ",
    }
    for digit, word in number_map.items():
        text = text.replace(digit, word)
    return text

def remove_punctuation(text):
    """
    Rimuove la punteggiatura sostituendola con spazi.

    Manteniamo solo caratteri alfanumerici e spazi.
    """
    return re.sub(r"[^\w\s]", " ", text)

def remove_extra_spaces(text):
    """
    Normalizza gli spazi multipli e rimuove eventuali spazi iniziali/finali.
    """
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text):
    """
    Suddivide il testo in token usando il tokenizer di NLTK.
    """
    return word_tokenize(text)

def remove_stop_words(tokens):
    """
    Elimina le stopwords inglesi, cioè parole molto frequenti e
    generalmente poco informative per il retrieval.
    """
    return [token for token in tokens if token not in stop_words]

def remove_single_characters(tokens):
    """
    Rimuove i token di lunghezza 1.

    È una scelta semplice per eliminare simboli residui o token poco utili.
    """
    return [token for token in tokens if len(token) > 1]

def apply_stemming(tokens):
    """
    Applica Porter stemming a ogni token.

    Lo stemming riduce parole morfologicamente simili a una radice comune.
    Esempio:
        'images' -> 'imag'
        'welcome' -> 'welcom'
    """
    return [stemmer.stem(token) for token in tokens]

def preprocess(text, is_query=False):
    """
    Applica l'intera pipeline di preprocessing a un documento o a una query.

    Parameters
    ----------
    text : str
        Testo in input.
    is_query : bool
        Se False, il testo è trattato come documento e quindi viene
        anche rimossa l'eventuale intestazione.
        Se True, il testo è trattato come query e non viene rimossa
        nessuna intestazione.

    Returns
    -------
    list[str]
        Lista di token preprocessati.
    """
    if not is_query:
        text = remove_header(text)

    text = convert_lower_case(text)
    text = convert_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    tokens = tokenize(text)
    tokens = remove_stop_words(tokens)
    tokens = remove_single_characters(tokens)
    tokens = apply_stemming(tokens)

    return tokens


# ============================================================
# 3. Indexing utilities
# ============================================================

def build_tokenized_documents(documents):
    """
    Applica il preprocessing a tutti i documenti della collezione.

    Parameters
    ----------
    documents : list[str]
        Lista dei documenti grezzi.

    Returns
    -------
    dict[int, list[str]]
        Dizionario che associa a ogni docID la lista dei token preprocessati.
    """
    tokenized_documents = {}

    for doc_id, text in enumerate(documents):
        tokenized_documents[doc_id] = preprocess(text, is_query=False)

    return tokenized_documents


def build_inverted_index(tokenized_documents):
    """
    Costruisce un inverted index non posizionale.

    Struttura restituita
    --------------------
    dict[str, list[int]]
        termine -> posting list ordinata dei docID

    Idea
    ----
    In un indice non posizionale, se un termine compare più volte
    nello stesso documento, il docID deve comparire una sola volta
    nella sua posting list.

    Per questo, mentre scorriamo i token di un documento, teniamo
    traccia dei termini già visti nel documento corrente.
    """
    inverted_index = {}

    for doc_id, tokens in tokenized_documents.items():
        seen_terms = set()

        for token in tokens:
            # Se abbiamo già visto questo termine nello stesso documento,
            # non aggiungiamo di nuovo lo stesso docID.
            if token in seen_terms:
                continue

            seen_terms.add(token)

            if token not in inverted_index:
                inverted_index[token] = []

            inverted_index[token].append(doc_id)

    # Non facciamo una sort finale delle posting lists:
    # i documenti vengono già visitati in ordine crescente di docID,
    # quindi le posting lists risultano ordinate per costruzione.
    return inverted_index


def build_positional_index(tokenized_documents):
    """
    Costruisce un positional index.

    Struttura restituita
    --------------------
    dict[str, dict[int, list[int]]]
        termine -> {docID -> lista delle posizioni}

    Esempio
    -------
    {
        "caesar": {
            1: [2],
            2: [0],
            4: [5]
        }
    }

    Nota
    ----
    Qui NON eliminiamo i duplicati del termine nello stesso documento,
    perché ogni occorrenza deve essere registrata con la sua posizione.
    """
    positional_index = {}

    for doc_id, tokens in tokenized_documents.items():
        for position, token in enumerate(tokens):
            if token not in positional_index:
                positional_index[token] = {}

            if doc_id not in positional_index[token]:
                positional_index[token][doc_id] = []

            positional_index[token][doc_id].append(position)

    return positional_index


def build_document_frequency(inverted_index):
    """
    Calcola la document frequency di ciascun termine.

    La document frequency di un termine è il numero di documenti
    in cui il termine compare almeno una volta.

    Parameters
    ----------
    inverted_index : dict[str, list[int]]
        Indice inverso non posizionale.

    Returns
    -------
    dict[str, int]
        termine -> document frequency
    """
    document_frequency = {}

    for term, posting_list in inverted_index.items():
        document_frequency[term] = len(posting_list)

    return document_frequency

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Merge, phrase query e query processing

In questa sezione costruiamo il cuore algoritmico del sistema di retrieval.

Dopo aver pre-elaborato la collezione e costruito gli indici, dobbiamo infatti definire come una query venga realmente interpretata ed eseguita.

In particolare introdurremo quattro componenti fondamentali:

- operazioni di **merge** su posting lists ordinate per implementare `AND`, `OR` e la negazione locale
- supporto alle **phrase query** tramite **positional index**
- una fase di **analisi lessicale e parsing** della query
- la procedura completa di **query execution**

L’obiettivo è passare da semplici strutture dati statiche a un sistema capace di interpretare query con:
- termini singoli
- frasi tra virgolette
- operatori booleani
- parentesi

Questa parte è fondamentale perché mostra che il retrieval non dipende solo dall’indice, ma anche dal modo in cui il sistema decide di **analizzare**, **rappresentare** e **valutare** le query dell’utente.

**IMPORTANTE**: Le parentesi sono supportate per raggruppare clausole complete, non per esprimere una grammatica booleana arbitrariamente annidata.

In [ ]:
# ============================================================
# 4. Merge utilities on sorted postings lists
# ============================================================

def intersect_sorted(a, b):
    """
    Interseca due posting lists ordinate.

    Parameters
    ----------
    a, b : list[int]
        Liste ordinate di docID.

    Returns
    -------
    list[int]
        Lista ordinata dei docID presenti in entrambe le liste.

    Idea
    ----
    Usiamo due puntatori, uno per ciascuna lista.

    - Se i due docID coincidono, il documento appartiene all'intersezione.
    - Se un docID è più piccolo dell'altro, allora quel docID non potrà
      più comparire nella seconda lista, perché le liste sono ordinate.
      Possiamo quindi avanzare il puntatore della lista con il docID minore.

    Questo è l'algoritmo classico per implementare AND tra posting lists.
    """
    i, j = 0, 0
    result = []

    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            # Lo stesso docID compare in entrambe le posting lists:
            # il documento soddisfa entrambe le condizioni.
            result.append(a[i])
            i += 1
            j += 1

        elif a[i] < b[j]:
            # a[i] è più piccolo di b[j].
            # Poiché b è ordinata, a[i] non potrà più comparire in b.
            # Quindi possiamo scartare a[i] e avanzare in a.
            i += 1

        else:
            # b[j] è più piccolo di a[i].
            # Per lo stesso motivo, b[j] non potrà più comparire in a.
            # Quindi possiamo scartare b[j] e avanzare in b.
            j += 1

    return result


def union_sorted(a, b):
    """
    Calcola l'unione di due posting lists ordinate.

    Parameters
    ----------
    a, b : list[int]
        Liste ordinate di docID.

    Returns
    -------
    list[int]
        Lista ordinata dei docID presenti in almeno una delle due liste.

    Idea
    ----
    Anche qui usiamo due puntatori.

    - Se i docID coincidono, li aggiungiamo una sola volta.
    - Se uno è più piccolo, lo aggiungiamo al risultato e avanziamo
      nella lista corrispondente.
    - Alla fine aggiungiamo gli eventuali elementi rimasti.

    Questa è l'operazione base per implementare OR.
    """
    i, j = 0, 0
    result = []

    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            # Il documento è presente in entrambe le liste.
            # Nell'unione deve comparire una sola volta.
            result.append(a[i])
            i += 1
            j += 1

        elif a[i] < b[j]:
            # a[i] è il più piccolo tra i due docID correnti:
            # deve comparire nell'unione.
            result.append(a[i])
            i += 1

        else:
            # b[j] è il più piccolo tra i due docID correnti:
            # deve comparire nell'unione.
            result.append(b[j])
            j += 1

    # Se una delle due liste non è finita,
    # i suoi elementi rimanenti vanno tutti nell'unione.
    while i < len(a):
        result.append(a[i])
        i += 1

    while j < len(b):
        result.append(b[j])
        j += 1

    return result


def difference_sorted(candidate_docs, docs_to_remove):
    """
    Calcola la differenza tra due liste ordinate:
        candidate_docs - docs_to_remove

    Parameters
    ----------
    candidate_docs : list[int]
        Documenti candidati da mantenere.
    docs_to_remove : list[int]
        Documenti da escludere.

    Returns
    -------
    list[int]
        Lista ordinata dei documenti che restano dopo l'esclusione.

    Idea
    ----
    Questa funzione implementa una negazione locale.

    Per esempio:
        A AND NOT B
    viene interpretato come:
        docs(A) - docs(B)

    Quindi NON stiamo costruendo il complemento dell'intera collezione.
    Stiamo solo togliendo da un insieme candidato positivo
    i documenti che soddisfano la parte negata.
    """
    i, j = 0, 0
    result = []

    while i < len(candidate_docs) and j < len(docs_to_remove):
        if candidate_docs[i] == docs_to_remove[j]:
            # Il documento compare anche tra quelli da rimuovere:
            # non deve andare nel risultato.
            i += 1
            j += 1

        elif candidate_docs[i] < docs_to_remove[j]:
            # Il documento candidato è più piccolo del prossimo documento da rimuovere.
            # Quindi non deve essere escluso e lo possiamo mantenere.
            result.append(candidate_docs[i])
            i += 1

        else:
            # docs_to_remove[j] è più piccolo.
            # Proviamo ad avanzare nei documenti da rimuovere
            # finché non raggiungiamo o superiamo candidate_docs[i].
            j += 1

    # Se restano documenti candidati, non sono più rimovibili:
    # li aggiungiamo tutti.
    while i < len(candidate_docs):
        result.append(candidate_docs[i])
        i += 1

    return result


# ============================================================
# 5. Phrase query support
# ============================================================

def phrase_query_docs(phrase_tokens, positional_index):
    """
    Restituisce i docID dei documenti che contengono esattamente la frase.

    Parameters
    ----------
    phrase_tokens : list[str]
        Lista dei token della frase, già preprocessati.
        Esempio: ["file", "format"]

    positional_index : dict
        Indice posizionale nel formato:
            termine -> {doc_id: [posizioni]}

        Esempio:
            "file"   -> {3: [7, 20], 8: [2]}
            "format" -> {3: [8], 8: [10]}

    Returns
    -------
    list[int]
        Lista ordinata dei docID che contengono la frase esatta.

    Idea generale
    -------------
    Una phrase query non si risolve direttamente guardando tutto il testo
    dei documenti.

    Si procede in due passi:

    1. si trovano i documenti candidati, cioè quelli che contengono
       tutti i termini della frase
    2. solo su questi candidati si controlla se i termini compaiono
       nelle posizioni giuste, cioè consecutivamente e nell'ordine corretto

    Esempio:
        per la frase ["file", "format"]
        dobbiamo trovare almeno una posizione p tale che:
            - "file" compaia in p
            - "format" compaia in p + 1
    """
    if len(phrase_tokens) == 0:
        # Caso limite: frase vuota.
        # Non ha senso restituire documenti.
        return []

    if len(phrase_tokens) == 1:
        # Una frase di un solo termine coincide semplicemente
        # con la posting list di quel termine.
        #
        # Usiamo il positional index e prendiamo solo i docID.
        # I docID sono già ordinati per costruzione.
        return list(positional_index.get(phrase_tokens[0], {}).keys())

    postings_per_term = []

    for term in phrase_tokens:
        postings_for_term = positional_index.get(term, {})

        # Se anche uno solo dei termini non compare nell'indice,
        # la frase non può comparire in nessun documento.
        if len(postings_for_term) == 0:
            return []

        postings_per_term.append(postings_for_term)

    # ------------------------------------------------------------
    # Fase 1: trovare i documenti candidati
    # ------------------------------------------------------------
    #
    # Per ogni termine abbiamo una struttura del tipo:
    #     term -> {doc_id: [positions]}
    #
    # A noi, in questa fase, interessano solo i docID.
    #
    # Partiamo dai documenti del primo termine e poi intersechiamo
    # progressivamente con quelli degli altri termini.
    #
    # Questo corrisponde a una AND query sui termini della frase.
    # Non usiamo OR/union, perché una phrase query richiede che
    # TUTTI i termini siano presenti nello stesso documento.
    common_docs = list(postings_per_term[0].keys())

    for postings_for_term in postings_per_term[1:]:
        common_docs = intersect_sorted(common_docs, list(postings_for_term.keys()))

    result_docs = []

    # ------------------------------------------------------------
    # Fase 2: controllo delle posizioni
    # ------------------------------------------------------------
    #
    # A questo punto common_docs contiene solo i documenti che
    # includono tutti i termini della frase.
    #
    # Ora dobbiamo verificare se, in almeno uno di questi documenti,
    # le posizioni sono allineate nel modo corretto.
    for doc_id in common_docs:
        # Posizioni del primo termine nel documento corrente.
        #
        # Esempio:
        #   "file" -> [3, 10, 25]
        #
        # Ognuna di queste posizioni può essere un possibile inizio
        # della frase.
        first_positions = postings_per_term[0][doc_id]

        # Per i termini successivi convertiamo le liste di posizioni in set.
        #
        # Perché?
        # Perché poi dovremo fare molti controlli del tipo:
        #   "la posizione start_pos + 1 esiste?"
        #   "la posizione start_pos + 2 esiste?"
        #
        # Su un set questo test è molto rapido.
        later_position_sets = [set(postings[doc_id]) for postings in postings_per_term[1:]]

        # Flag che indica se abbiamo trovato almeno una occorrenza valida
        # della frase nel documento corrente.
        found = False

        # Proviamo ogni posizione del primo termine come possibile inizio.
        for start_pos in first_positions:
            match = True

            # Se la frase inizia in start_pos, allora:
            # - il secondo termine deve stare in start_pos + 1
            # - il terzo termine deve stare in start_pos + 2
            # - ...
            for offset, pos_set in enumerate(later_position_sets, start=1):
                if start_pos + offset not in pos_set:
                    # Appena un termine manca nella posizione attesa,
                    # questa partenza non è valida.
                    match = False
                    break

            if match:
                # Abbiamo trovato almeno una occorrenza corretta
                # della frase nel documento.
                found = True
                break

        if found:
            result_docs.append(doc_id)

    return result_docs


## Dal testo della query al risultato finale

Dopo aver costruito:

- l'**inverted index**, utile per recuperare i documenti associati a un termine
- il **positional index**, utile per verificare frasi esatte

possiamo passare al **query processing**, cioè alla fase in cui una query testuale dell'utente viene:

1. letta come stringa
2. scomposta in parti elementari
3. interpretata secondo la grammatica supportata
4. valutata usando le posting lists e, quando serve, le posizioni

### Idee chiave

Nel nostro notebook una query può contenere:

- **termini singoli**, ad esempio `graphic`
- **frasi tra virgolette**, ad esempio `"file format"`
- operatori booleani `AND`, `OR`, `NOT`
- parentesi, ma solo per raggruppare **clausole complete**

La semantica adottata è controllata:

- la query completa è vista come una **OR di clausole**
- ogni clausola è vista come una **AND di unità**
- `NOT` è trattato come **filtro locale**
- ogni clausola deve contenere almeno una parte positiva

Questa scelta rende il sistema più semplice da capire e più vicino a una implementazione concreta basata su posting lists ordinate.

---

## Come vengono chiamati i metodi

L'esecuzione di una query segue questa catena di chiamate:

### 1. `execute_boolean_query(...)`
È il punto di ingresso principale.

Questa funzione:
- riceve la query come stringa
- avvia il parsing
- valuta le clausole
- unisce i risultati finali con `OR`

### 2. `lex_query(...)`
Fa una **analisi lessicale** molto semplice.

Trasforma la stringa in una lista di token, preservando:
- parentesi
- operatori booleani
- frasi tra virgolette

Ad esempio:

```python
(graphic AND NOT "file format") OR imag
```

diventa:

```python
['(', 'graphic', 'AND', 'NOT', '"file format"', ')', 'OR', 'imag']
```

### 3. `split_top_level_or_from_tokens(...)`
Divide la query nelle clausole separate da `OR` **al livello più esterno**.

Questo è importante perché un `OR` dentro parentesi non deve spezzare la query esterna.

### 4. `parse_and_clause_from_tokens(...)`
Analizza una singola clausola e la trasforma in una struttura del tipo:

```python
{
    "positive": [...],
    "negative": [...]
}
```

Per farlo usa anche:

- `strip_balanced_outer_parentheses(...)`
- `validate_clause_tokens(...)`

Queste funzioni servono a:
- togliere parentesi esterne inutili
- controllare che la clausola rispetti la grammatica supportata

### 5. `evaluate_clause(...)`
Valuta una clausola già parsata.

Il comportamento è:

- recupera i documenti associati alle unità positive
- interseca i risultati con `AND`
- rimuove i documenti associati alle unità negative

Questa funzione usa direttamente:

- `intersect_sorted(...)`
- `difference_sorted(...)`

### 6. `get_docs_for_unit(...)`
Recupera i documenti associati a una singola unità della query.

Qui ci sono due casi:

- se l'unità è un **termine singolo**, usa l'**inverted index**
- se l'unità è una **frase**, usa il **positional index**

Per distinguere i due casi usa:

- `is_phrase(...)`

e, nel caso delle frasi:

- `phrase_query_docs(...)`

### 7. `phrase_query_docs(...)`
Verifica se una frase compare davvero in un documento.

La logica è in due passi:

1. trovare i documenti che contengono tutti i termini della frase
2. controllare se quei termini compaiono in posizioni consecutive e nell'ordine corretto

Qui entra in gioco il **positional index**.

---

## Riassunto operativo

In forma compatta, il flusso è questo:

```python
execute_boolean_query
    -> lex_query
    -> split_top_level_or_from_tokens
    -> parse_and_clause_from_tokens
        -> strip_balanced_outer_parentheses
        -> validate_clause_tokens
    -> evaluate_clause
        -> get_docs_for_unit
            -> is_phrase
            -> phrase_query_docs   # se l'unità è una phrase query
        -> intersect_sorted
        -> difference_sorted
    -> union_sorted
```

---

## Osservazione importante

Questa parte del notebook non introduce ancora un parser booleano completamente generale.

Stiamo invece implementando un linguaggio di query:

- più ristretto
- più controllato



In [ ]:
# ============================================================
# 6. Query unit handling
# ============================================================

def is_phrase(unit):
    """
    Verifica se una unità della query è una frase tra virgolette.

    Examples
    --------
    'graphic' -> False
    '"file format"' -> True
    """
    return len(unit) >= 2 and unit[0] == '"' and unit[-1] == '"'


def get_docs_for_unit(unit, postings, positional_index):
    """
    Restituisce i documenti associati a una singola unità di query.

    Una unità può essere:
    - un termine singolo
    - una frase tra virgolette

    Parameters
    ----------
    unit : str
        Unità della query.
    postings : dict[str, list[int]]
        Inverted index non posizionale.
    positional_index : dict
        Positional index.

    Returns
    -------
    list[int]
        Lista ordinata dei documenti che soddisfano la unità.

    Idea
    ----
    - Se l'unità è una frase, usiamo il positional index.
    - Se l'unità è un termine singolo, usiamo l'inverted index standard.
    """
    if is_phrase(unit):
        # Rimuoviamo le virgolette e preprocessiamo il contenuto della frase
        # nello stesso modo in cui preprocessiamo le query.
        phrase_text = unit[1:-1]
        phrase_tokens = preprocess(phrase_text, is_query=True)
        return phrase_query_docs(phrase_tokens, positional_index)

    else:
        # Un termine singolo deve essere preprocessato prima di cercarlo
        # nell'indice, altrimenti rischiamo di usare una forma diversa
        # da quella memorizzata.
        processed_tokens = preprocess(unit, is_query=True)

        if len(processed_tokens) == 0:
            # Il termine può sparire dopo il preprocessing
            # (per esempio se è una stopword).
            return []

        return postings.get(processed_tokens[0], [])


# ============================================================
# 7. Query tokenization and parsing
# ============================================================

def lex_query(query):
    """
    Tokenizza la query preservando:
    - parentesi
    - frasi tra virgolette
    - operatori booleani
    - termini normali

    Parameters
    ----------
    query : str
        Query in input.

    Returns
    -------
    list[str]
        Lista dei token lessicali della query.

    Example
    -------
    '(graphic AND NOT "file format") OR imag'
    ->
    ['(', 'graphic', 'AND', 'NOT', '"file format"', ')', 'OR', 'imag']

    Nota
    ----
    Questa è una analisi lessicale molto semplice:
    non interpreta ancora la struttura logica della query,
    ma la divide solo in unità elementari.
    """
    pattern = r'"[^"]+"|\(|\)|AND|OR|NOT|[^\s()]+'
    return re.findall(pattern, query, flags=re.IGNORECASE)


def split_top_level_or_from_tokens(tokens):
    """
    Divide la query in clausole separate da OR a livello più esterno.

    Parameters
    ----------
    tokens : list[str]
        Lista dei token lessicali.

    Returns
    -------
    list[list[str]]
        Lista di clausole, ognuna rappresentata come lista di token.

    Idea
    ----
    La query viene interpretata come una OR di clausole.

    Però un OR dentro parentesi non deve spezzare la query esterna.
    Per questo teniamo traccia della profondità delle parentesi:
    solo gli OR trovati a profondità 0 separano davvero due clausole.
    """
    parts = []
    current = []
    depth = 0

    for token in tokens:
        upper = token.upper()

        if token == "(":
            depth += 1
            current.append(token)

        elif token == ")":
            depth -= 1
            current.append(token)

        elif upper == "OR" and depth == 0:
            # Questo OR è di livello esterno:
            # chiude la clausola corrente e ne apre una nuova.
            parts.append(current)
            current = []

        else:
            current.append(token)

    if current:
        parts.append(current)

    return parts


def strip_balanced_outer_parentheses(tokens):
    """
    Rimuove una coppia esterna di parentesi se racchiude tutta la clausola.

    Parameters
    ----------
    tokens : list[str]
        Token della clausola.

    Returns
    -------
    list[str]
        Clausola senza la coppia più esterna di parentesi,
        se quella coppia racchiude davvero tutta l'espressione.

    Example
    -------
    ['(', 'graphic', 'AND', 'file', ')'] -> ['graphic', 'AND', 'file']

    Idea
    ----
    Se la clausola è del tipo:
        ( ... )
    e le parentesi esterne bilanciano tutta la sequenza,
    possiamo toglierle senza cambiare il significato.
    """
    while len(tokens) >= 2 and tokens[0] == "(" and tokens[-1] == ")":
        depth = 0
        fully_enclosed = True

        for i, tok in enumerate(tokens):
            if tok == "(":
                depth += 1
            elif tok == ")":
                depth -= 1

            # Se torniamo a profondità zero prima della fine,
            # allora la parentesi iniziale non racchiude tutta la clausola.
            if depth == 0 and i < len(tokens) - 1:
                fully_enclosed = False
                break

        if fully_enclosed:
            tokens = tokens[1:-1]
        else:
            break

    return tokens


def validate_clause_tokens(tokens):
    """
    Controlla che una clausola rispetti la grammatica supportata.

    Forma ammessa
    -------------
        operand (AND operand)*

    dove un operand può essere:
    - unit
    - NOT unit

    Inoltre, la clausola deve contenere almeno una unità positiva.

    Parameters
    ----------
    tokens : list[str]
        Token della clausola.

    Raises
    ------
    ValueError
        Se la clausola non è ben formata.

    Nota
    ----
    Questa validazione riflette la semantica scelta nel laboratorio:
    non supportiamo un NOT globale della collezione.
    """
    if len(tokens) == 0:
        raise ValueError("Empty clause.")

    positive_count = 0
    i = 0
    expect_operand = True

    while i < len(tokens):
        tok = tokens[i]
        upper = tok.upper()

        if expect_operand:
            if upper == "AND":
                raise ValueError("Unexpected AND.")

            if tok in ("(", ")"):
                raise ValueError(
                    "Parentheses are only allowed around full clauses, not arbitrary nesting inside AND clauses."
                )

            if upper == "NOT":
                i += 1

                if i >= len(tokens):
                    raise ValueError("NOT must be followed by a term or a phrase.")

                next_tok = tokens[i]

                if next_tok.upper() in {"AND", "OR", "NOT"} or next_tok in {"(", ")"}:
                    raise ValueError("Invalid token after NOT.")
            else:
                # Abbiamo trovato una unità positiva.
                positive_count += 1

            expect_operand = False
            i += 1

        else:
            if upper != "AND":
                raise ValueError("Only AND is allowed inside a clause.")

            expect_operand = True
            i += 1

    if expect_operand:
        raise ValueError("Clause cannot end with AND.")

    if positive_count == 0:
        raise ValueError(
            "Each clause must contain at least one positive unit. Global NOT is not supported."
        )


def parse_and_clause_from_tokens(tokens):
    """
    Analizza una clausola e la trasforma nel formato:

        {
            "positive": [...],
            "negative": [...]
        }

    Parameters
    ----------
    tokens : list[str]
        Token della clausola.

    Returns
    -------
    dict
        Clausola separata in unità positive e negative.

    Idea
    ----
    Una clausola viene interpretata come:
    - intersezione delle unità positive
    - seguita dalla sottrazione locale delle unità negative

    Quindi separiamo esplicitamente i due gruppi.
    """
    tokens = strip_balanced_outer_parentheses(tokens)
    validate_clause_tokens(tokens)

    positive = []
    negative = []

    i = 0
    while i < len(tokens):
        tok = tokens[i]
        upper = tok.upper()

        if upper == "AND":
            i += 1
            continue

        if upper == "NOT":
            negative.append(tokens[i + 1])
            i += 2
        else:
            positive.append(tok)
            i += 1

    return {"positive": positive, "negative": negative}


# ============================================================
# 8. Query execution
# ============================================================

def evaluate_clause(clause, postings, positional_index):
    """
    Valuta una singola clausola.

    La clausola ha la forma:
        positive_1 AND positive_2 AND ... AND NOT negative_1 AND ...

    Parameters
    ----------
    clause : dict
        Clausola nel formato:
            {
                "positive": [...],
                "negative": [...]
            }
    postings : dict
        Inverted index non posizionale.
    positional_index : dict
        Positional index.

    Returns
    -------
    list[int]
        Lista ordinata dei documenti che soddisfano la clausola.

    Strategy
    --------
    1. Recuperiamo i documenti per tutte le unità positive.
    2. Intersechiamo i risultati positivi.
    3. Rimuoviamo i documenti che soddisfano le unità negative.

    Questa è la semantica locale di NOT adottata nel laboratorio.
    """
    # Recuperiamo le posting lists o i risultati di phrase query
    # associati alle unità positive.
    positive_lists = [
        get_docs_for_unit(unit, postings, positional_index)
        for unit in clause["positive"]
    ]

    # Intersecare prima le liste più piccole è in genere più efficiente.
    positive_lists = sorted(positive_lists, key=len)

    current_result = positive_lists[0]

    for doc_list in positive_lists[1:]:
        current_result = intersect_sorted(current_result, doc_list)

    # Ora togliamo dal risultato positivo i documenti che soddisfano
    # le unità negative.
    for unit in clause["negative"]:
        docs_to_remove = get_docs_for_unit(unit, postings, positional_index)
        current_result = difference_sorted(current_result, docs_to_remove)

    return current_result


def execute_boolean_query(query, postings, positional_index, verbose=True):
    """
    Esegue una query booleana nel linguaggio supportato dal laboratorio.

    Esempi supportati
    -----------------
    graphic AND file
    graphic AND NOT file
    graphic AND NOT "file format"
    (graphic AND NOT "file format") OR imag

    Parameters
    ----------
    query : str
        Query in input.
    postings : dict
        Inverted index non posizionale.
    positional_index : dict
        Positional index.
    verbose : bool
        Se True, stampa le fasi intermedie.

    Returns
    -------
    list[int]
        Lista ordinata dei documenti restituiti dalla query.

    Pipeline
    --------
    1. Analisi lessicale della query
    2. Suddivisione in clausole OR di livello esterno
    3. Parsing di ciascuna clausola
    4. Valutazione delle clausole
    5. Unione finale dei risultati
    """
    tokens = lex_query(query)

    # Separiamo la query in clausole collegate da OR esterno.
    or_clauses_tokens = split_top_level_or_from_tokens(tokens)

    # Trasformiamo ogni clausola nella struttura:
    # {"positive": [...], "negative": [...]}
    parsed_clauses = [
        parse_and_clause_from_tokens(clause_tokens)
        for clause_tokens in or_clauses_tokens
    ]

    if verbose:
        print("Lexed tokens:")
        print(tokens)

        print("\nParsed clauses:")
        for i, clause in enumerate(parsed_clauses, start=1):
            print(f"Clause {i}: {clause}")
        print()

    clause_results = []

    for i, clause in enumerate(parsed_clauses, start=1):
        result = evaluate_clause(clause, postings, positional_index)
        clause_results.append(result)

        if verbose:
            print(f"Clause {i} result ({len(result)} docs):")
            print(result[:50])
            print()

    # La query completa è una OR di clausole:
    # quindi il risultato finale è l'unione dei risultati delle clausole.
    final_result = []

    for result in clause_results:
        final_result = union_sorted(final_result, result)

    if verbose:
        print("Final result:")
        print(final_result[:100])
        print(f"Total retrieved documents: {len(final_result)}")

    return final_result

## Costruzione della collezione e degli indici

Ora mettiamo insieme tutte le componenti definite fin qui e costruiamo il sistema su una collezione reale.

In questa sezione:

- carichiamo un sottoinsieme del dataset **20 Newsgroups**
- applichiamo il **preprocessing** a tutti i documenti
- costruiamo sia l’**inverted index** sia il **positional index**
- calcoliamo la **document frequency** dei termini
- ispezioniamo alcuni esempi per verificare che la pipeline stia funzionando correttamente

Questo passaggio è importante perché trasforma le utility definite in precedenza in un sistema effettivamente interrogabile.

In [ ]:
# ============================================================
# 9. Build the collection and the indexes
# ============================================================

# Per semplicità lavoriamo su una sola categoria del dataset.
categories = ["comp.graphics"]

# 1. Carichiamo i documenti grezzi.
documents = load_collection(categories=categories, subset="train")

# 2. Applichiamo il preprocessing a tutta la collezione.
tokenized_documents = build_tokenized_documents(documents)

# 3. Costruiamo l'indice invertito non posizionale.
postings = build_inverted_index(tokenized_documents)

# 4. Costruiamo anche l'indice posizionale per supportare le phrase query.
positional_index = build_positional_index(tokenized_documents)
# NOTA: ma non potremmo derivare la struttura postings attraverso una analisi di positional_index?

# 5. Calcoliamo la document frequency di ciascun termine.
document_frequency = build_document_frequency(postings)

print("Selected categories:", categories)
print("Number of documents:", len(documents))
print("Vocabulary size:", len(postings))

# Mostriamo un esempio di documento grezzo per osservare il rumore iniziale.
print("\nFirst raw document:\n")
print(documents[0][:1200])

print("\n" + "=" * 80 + "\n")

# Mostriamo lo stesso documento dopo il preprocessing.
print("First preprocessed document:\n")
print(tokenized_documents[0][:80])

# Ispezioniamo alcune posting lists per verificare la struttura dell'indice.
print("\nSome example postings:\n")
for term in ["graphic", "imag", "file", "format", "window"]:
    print(f"{term:12s} -> {postings.get(term, [])[:30]}")

Selected categories: ['comp.graphics']
Number of documents: 584
Vocabulary size: 8603

First raw document:

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
    

## Esempi di query

A questo punto possiamo usare il sistema costruito nel notebook per eseguire alcune query rappresentative.

Gli esempi scelti servono a mostrare progressivamente:

- una query con soli termini positivi
- una query con **negazione locale**
- una query con **phrase query** all’interno di una clausola
- una query con **parentesi** e combinazione tramite `OR`

Questa sezione è utile perché permette di osservare non solo il risultato finale, ma anche le fasi intermedie del **parsing** e del **query processing**.

**IMPORTANTE**: Le parentesi sono ammesse solo per raggruppare clausole complete collegate da `OR`; non supportiamo una grammatica booleana completamente arbitraria con annidamento libero di `AND`, `OR` e `NOT`.

In [ ]:
# ============================================================
# 10. Query examples
# ============================================================

# Alcuni esempi rappresentativi del linguaggio supportato:
# - AND tra termini
# - NOT come filtro locale
# - phrase query tra virgolette
# - combinazione di clausole con OR e parentesi
example_queries = [
    'graphic',
    '"file format"',
    'graphic AND file',
    'graphic AND NOT file',
    'graphic AND NOT "file format"',
    '(graphic AND NOT "file format") OR imag'
]

# Eseguiamo ogni query in modalità verbosa per osservare:
# - tokenizzazione lessicale
# - parsing in clausole
# - risultati intermedi delle clausole
# - risultato finale
for q in example_queries:
    print("\n" + "=" * 100)
    print("QUERY:", q)
    execute_boolean_query(q, postings, positional_index, verbose=True)


QUERY: graphic
Lexed tokens:
['graphic']

Parsed clauses:
Clause 1: {'positive': ['graphic'], 'negative': []}

Clause 1 result (190 docs):
[2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76, 77, 79, 83, 87, 88, 93, 101, 103, 104, 109, 118, 119, 120, 121, 124, 125, 130, 132, 134, 138, 142, 146, 148, 151, 154, 156, 163, 164, 166, 169]

Final result:
[2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76, 77, 79, 83, 87, 88, 93, 101, 103, 104, 109, 118, 119, 120, 121, 124, 125, 130, 132, 134, 138, 142, 146, 148, 151, 154, 156, 163, 164, 166, 169, 174, 182, 184, 188, 192, 194, 195, 199, 204, 205, 209, 212, 213, 215, 220, 224, 226, 231, 232, 234, 245, 247, 250, 258, 259, 263, 264, 271, 272, 275, 276, 277, 278, 284, 287, 288, 292, 302, 303, 305, 307, 312, 313, 314, 315, 316, 319, 322, 325, 327]
Total retrieved documents: 190

QUERY: "file format"
Lexed tokens:
['"file format"']

Parsed clauses:
Clause 1: {'positive': ['"file format"'], 'negative

## Ispezione dei risultati

Dopo aver verificato il funzionamento del parser e del query processing, è utile osservare direttamente alcuni documenti recuperati.

Questo passaggio è importante perché permette di collegare:

- la **semantica della query**
- il **comportamento dell’algoritmo**
- il **contenuto effettivo** dei documenti restituiti

In altre parole, non ci interessa solo sapere *quali* documenti vengono recuperati, ma anche capire *perché* risultano compatibili con la query.

In [ ]:
def print_document(doc_id, documents, max_chars=1200):
    """
    Stampa una porzione del documento originale associato a un docID.

    Parameters
    ----------
    doc_id : int
        Identificatore del documento.
    documents : list[str]
        Collezione dei documenti grezzi.
    max_chars : int
        Numero massimo di caratteri da mostrare.
    """
    print(f"Document {doc_id}\n")
    print(documents[doc_id][:max_chars])


# Scegliamo una query un po' più ricca, che combina:
# - un termine positivo
# - una negazione locale su una frase
# - una seconda clausola collegata con OR
query = '(graphic AND NOT "file format") OR imag'

# Eseguiamo la query senza stampa verbosa per concentrarci solo sui risultati finali.
results = execute_boolean_query(query, postings, positional_index, verbose=False)

print(f"Retrieved {len(results)} documents.\n")

# Mostriamo i primi documenti recuperati per una valutazione qualitativa.
for doc_id in results[:3]:
    print("=" * 80)
    print_document(doc_id, documents)

Retrieved 282 documents.

Document 0

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
              :           bbs.mirage@gilligan.tsoft.net  :
              :

## Esempi di query non supportate

Il linguaggio di query adottato in questo laboratorio è volutamente più restrittivo di un linguaggio booleano completamente generale.

In particolare, la negazione `NOT` viene trattata solo come **filtro locale** all’interno di una clausola che contiene almeno una unità positiva.

Per questo motivo, alcune query che sarebbero ammissibili in altri sistemi booleani **non sono supportate** qui.

Osservare questi casi è utile perché chiarisce meglio la semantica del linguaggio adottato e i limiti progettuali del sistema.

In [ ]:
unsupported_queries = [
    'NOT graphic',
    'imag OR NOT graphic',
    'NOT "file format"',
    'graphic AND (NOT file OR imag)'
]

for q in unsupported_queries:
    print("\n" + "=" * 100)
    print("UNSUPPORTED QUERY:", q)
    try:
        execute_boolean_query(q, postings, positional_index, verbose=True)
    except ValueError as e:
        print("Raised error:")
        print(e)


UNSUPPORTED QUERY: NOT graphic
Raised error:
Each clause must contain at least one positive unit. Global NOT is not supported.

UNSUPPORTED QUERY: imag OR NOT graphic
Raised error:
Each clause must contain at least one positive unit. Global NOT is not supported.

UNSUPPORTED QUERY: NOT "file format"
Raised error:
Each clause must contain at least one positive unit. Global NOT is not supported.

UNSUPPORTED QUERY: graphic AND (NOT file OR imag)
Raised error:
Parentheses are only allowed around full clauses, not arbitrary nesting inside AND clauses.


## Limiti della versione attuale

Il sistema costruito in questo notebook è molto più espressivo del prototipo del laboratorio precedente, ma mantiene ancora alcune restrizioni deliberate:

- non supporta una negazione globale della collezione
- non implementa una grammatica booleana completamente arbitraria
- non include query di prossimità
- non introduce ancora ranking o ordinamento per rilevanza

Queste restrizioni rendono il sistema più semplice da comprendere e più controllabile dal punto di vista didattico.

## Riassunto del laboratorio

In questo laboratorio abbiamo trasformato il prototipo booleano sviluppato nel notebook precedente in un sistema di retrieval più strutturato, più espressivo e più vicino alle scelte progettuali che si incontrano nei sistemi reali.

### Abbiamo introdotto
- una semantica di query più rigorosa
- una distinzione chiara tra:
  - **query completa**
  - **clausola**
  - **unità**
- una gestione controllata dell’operatore `NOT` come **filtro locale**
- il supporto a **termini**, **frasi tra virgolette** e **parentesi**

### Abbiamo costruito
- una pipeline di **preprocessing** per documenti e query
- un **inverted index** non posizionale
- un **positional index**
- le **document frequencies** dei termini

### Abbiamo implementato
- operazioni di **merge** su posting lists ordinate per `AND` e `OR`
- una differenza ordinata per realizzare la negazione locale
- il supporto alle **phrase query** tramite positional index
- una fase di **analisi lessicale**
- una fase di **parsing** della query
- una procedura completa di **query execution**

### Abbiamo anche chiarito che
il comportamento di un sistema booleano dipende non solo dall’indice, ma anche dalla **semantica del linguaggio di query** che scegliamo di supportare.

In particolare, in questo notebook:
- `NOT` non è interpretato come complemento globale della collezione
- le query sono viste come una **OR di clausole**
- ogni clausola è una **AND di unità**
- ogni clausola deve contenere almeno una unità positiva

## Messaggio chiave

Il retrieval booleano non è solo una questione di operatori logici applicati a liste di documenti.

È il risultato congiunto di:
- una scelta di **rappresentazione** dei documenti
- una struttura di **indicizzazione**
- una semantica del **linguaggio di query**
- una strategia di **query processing**

Capire come queste componenti interagiscono è essenziale per progettare sistemi di Information Retrieval più efficaci, più interpretabili e più vicini ai motori di ricerca reali.

## Limiti della versione attuale

Il sistema costruito in questo notebook è più espressivo del prototipo del laboratorio precedente, ma mantiene alcune restrizioni deliberate.

In particolare:

- `NOT` non è interpretato come complemento globale della collezione, ma come **filtro locale** su un insieme candidato positivo
- le query **non** seguono una grammatica booleana completamente generale
- le parentesi sono ammesse solo per raggruppare **clausole complete**
- non sono ancora supportate le **proximity queries**
- non è previsto alcun **ranking** dei risultati

Queste scelte non sono accidentali: servono a mantenere il sistema **implementabile, leggibile e didatticamente controllato**.

Più precisamente, il notebook non implementa l’intera algebra booleana sulle query, ma un **linguaggio di query semplificato** in cui:

- una query è una **OR di clausole**
- ogni clausola è una **AND di unità**
- ogni clausola deve contenere almeno una unità positiva
- `NOT` può comparire solo come negazione locale di un termine o di una frase

Questa restrizione rende più semplice il query processing e permette di concentrarsi sugli aspetti centrali del laboratorio:

- **inverted index**
- **positional index**
- **merge di posting lists**
- **phrase queries**

---

## Riassunto del laboratorio

In questo laboratorio abbiamo trasformato il prototipo booleano sviluppato nel notebook precedente in un sistema di retrieval più strutturato, più espressivo e più vicino alle scelte progettuali che si incontrano nei sistemi reali.

### Abbiamo introdotto

- una semantica di query più rigorosa
- una distinzione chiara tra **query**, **clausola** e **unità**
- una gestione controllata dell’operatore `NOT` come **filtro locale**
- il supporto a **termini**, **frasi tra virgolette** e **parentesi**

### Abbiamo costruito

- una pipeline di **preprocessing** per documenti e query
- un **inverted index** non posizionale
- un **positional index**
- le **document frequencies** dei termini

### Abbiamo implementato

- operazioni di **merge** su posting lists ordinate per `AND` e `OR`
- una differenza ordinata per realizzare la negazione locale
- il supporto alle **phrase queries** tramite positional index
- una fase di **analisi lessicale**
- una fase di **parsing** della query
- una procedura completa di **query execution**

### Messaggio chiave

Il retrieval booleano non è solo una questione di operatori logici applicati a liste di documenti.

È il risultato congiunto di:

- una scelta di **rappresentazione** dei documenti
- una struttura di **indicizzazione**
- una semantica del **linguaggio di query**
- una strategia di **query processing**

Capire come queste componenti interagiscono è essenziale per progettare sistemi di Information Retrieval più efficaci, più interpretabili e più vicini ai motori di ricerca reali.

---

## Esercizi finali

### Esercizio 1 — Aggiungere le proximity queries

Estendi il sistema in modo da supportare query di vicinanza tra termini, ad esempio:

- `graphic NEAR/3 file`
- `"file" NEAR/5 "format"`

Usa il **positional index** per verificare se due termini compaiono entro una distanza massima `k`.
